In [1]:
!nvidia-smi

Sat Jul  4 05:02:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   47G   66G  42% /


In [3]:
import psutil, subprocess

print("═══ RAM Check ═══")
print(f"Total: {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"Available: {psutil.virtual_memory().available / 1e9:.2f} GB")

print("\n═══ Disk Check ═══")
subprocess.run(["df", "-h", "/content"])

═══ RAM Check ═══
Total: 13.61 GB
Available: 12.58 GB

═══ Disk Check ═══


CompletedProcess(args=['df', '-h', '/content'], returncode=0)

In [4]:
import os
print(os.path.exists("/content/kaggle.json"))

False


In [5]:
!pip install pyspark
!pip install nltk
!pip install gensim
!pip install scikit-learn
!pip install tensorflow
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.7 MB/s eta 0:00:00


In [6]:
!pip install transformers torch
# If on Colab with GPU (recommended — BERT on CPU is very slow):
# Runtime > Change runtime type > T4 GPU

In [7]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("SentimentAnalysisResearch") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [8]:
from transformers import BertTokenizer, BertModel
import torch

# Load once on the driver — will be re-loaded per executor in the UDF
BERT_MODEL_NAME = 'bert-base-uncased'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Pre-download the model weights to Colab cache so the UDF doesn't re-download
tokenizer_bert = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
_ = BertModel.from_pretrained(BERT_MODEL_NAME)  # triggers download/cache
print("BERT model cached successfully.")

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model cached successfully.


In [9]:
!pip install kaggle

In [10]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"

In [11]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 129MB/s] 



In [12]:
!unzip -q imdb-dataset-of-50k-movie-reviews.zip

In [13]:
df_imdb = spark.read.csv(
    "IMDB Dataset.csv",
    header=True,
    inferSchema=True,
     multiLine=True,
    quote='"',
    escape='"'
)

df_imdb.show(5, truncate=False)
df_imdb.printSchema()

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
# ══════════════════════════════════════════════
# ── Dataset ko 50% reduce karo ──
# ══════════════════════════════════════════════

# Original size check
total_rows = df_imdb.count()
print(f"Original dataset size: {total_rows} rows")

# ── Stratified sampling — positive aur negative dono 50% rakho ──
# Isse class balance maintain rahega
df_imdb = df_imdb.sampleBy(
    col="sentiment",
    fractions={"positive": 0.5, "negative": 0.5},
    seed=42
)

# Reduced size verify karo
reduced_rows = df_imdb.count()
print(f"Reduced dataset size: {reduced_rows} rows")

# Class distribution check — balance verify karo
print("\nClass distribution after reduction:")
df_imdb.groupBy("sentiment").count().show()

Original dataset size: 50000 rows
Reduced dataset size: 25078 rows

Class distribution after reduction:
+---------+-----+
|sentiment|count|
+---------+-----+
| positive|12469|
| negative|12609|
+---------+-----+



In [15]:
from pyspark.sql.functions import col, lower, regexp_replace, trim
from pyspark.ml.feature import StringIndexer

# ── Step 1: Base cleaning — sirf HTML aur extra spaces (sab ke liye common) ──
df_base = df_imdb \
    .withColumn("review_base",
        regexp_replace(col("review"), "<br\\s*/?>", " ")) \
    .withColumn("review_base",
        trim(regexp_replace(col("review_base"), "\\s+", " ")))

# ── Step 2: Traditional techniques ke liye (TF-IDF, Word2Vec, GloVe, FastText) ──
# Lowercase + special characters remove
df_base = df_base \
    .withColumn("review_traditional",
        lower(col("review_base"))) \
    .withColumn("review_traditional",
        regexp_replace(col("review_traditional"), "[^a-z\\s]", ""))

# ── Step 3: BERT ke liye alag column — kuch mat badlo ──
df_base = df_base \
    .withColumn("review_bert", col("review_base"))

# ── Step 4: Label encoding ──
indexer = StringIndexer(
    inputCol="sentiment",
    outputCol="label",
    stringOrderType="alphabetAsc"  # deterministic: negative=0, positive=1
)
df_labeled = indexer.fit(df_base).transform(df_base)

# Verify
df_labeled.select("review_bert", "review_traditional", "sentiment", "label") \
    .show(5, truncate=60)

+------------------------------------------------------------+------------------------------------------------------------+---------+-----+
|                                                 review_bert|                                          review_traditional|sentiment|label|
+------------------------------------------------------------+------------------------------------------------------------+---------+-----+
|Basically there's a family where a little boy (Jake) thin...|basically theres a family where a little boy jake thinks ...| negative|  0.0|
|This show was an amazing, fresh & innovative idea in the ...|this show was an amazing fresh  innovative idea in the s ...| negative|  0.0|
|Phil the Alien is one of those quirky films where the hum...|phil the alien is one of those quirky films where the hum...| negative|  0.0|
|So im not a big fan of Boll's work but then again not man...|so im not a big fan of bolls work but then again not many...| negative|  0.0|
|Some films just sim

In [16]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# ── Step 1: Tokenization — review_traditional column use karo ──
tokenizer = Tokenizer(inputCol="review_traditional", outputCol="words")
df_tokenized = tokenizer.transform(df_labeled)

# ── Step 2: Custom stop words — negation words preserve karo ──
default_sw = StopWordsRemover.loadDefaultStopWords("english")

# Yeh words sentiment ke liye important hain, isliye list se hatao
keep_words = {
    "not", "no", "never", "nor", "neither",
    "but", "however", "although", "though",
    "barely", "hardly", "scarcely", "without"
}
custom_stopwords = [w for w in default_sw if w not in keep_words]

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words",
    stopWords=custom_stopwords
)
df_preprocessed = remover.transform(df_tokenized)

# Verify — teen columns dikhenge
df_preprocessed.select(
    "review_traditional", "filtered_words", "review_bert"
).show(3, truncate=60)

+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|                                          review_traditional|                                              filtered_words|                                                 review_bert|
+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|basically theres a family where a little boy jake thinks ...|[basically, theres, family, little, boy, jake, thinks, th...|Basically there's a family where a little boy (Jake) thin...|
|this show was an amazing fresh  innovative idea in the s ...|[show, amazing, fresh, , innovative, idea, first, aired, ...|This show was an amazing, fresh & innovative idea in the ...|
|phil the alien is one of those quirky films where the hum...|[phil, alien,

In [17]:
# ── Sirf objects define karo, fit mat karo abhi ──
from pyspark.ml.feature import CountVectorizer, IDF

cv = CountVectorizer(
    inputCol="filtered_words",
    outputCol="rawFeatures",
    vocabSize=5000,   # 1000 se 5000 kiya — better accuracy
    minDF=5.0
)

idf = IDF(inputCol="rawFeatures", outputCol="tfidf_features")

print("TF-IDF objects defined. Fitting will happen after train/test split.")

TF-IDF objects defined. Fitting will happen after train/test split.


In [18]:
# ── Sirf object define karo, fit mat karo abhi ──
from pyspark.ml.feature import Word2Vec

word2vec = Word2Vec(
    vectorSize=100,
    minCount=5,
    inputCol="filtered_words",
    outputCol="word2vec_features",
    maxIter=3
)

print("Word2Vec object defined. Fitting will happen after train/test split.")

Word2Vec object defined. Fitting will happen after train/test split.


In [19]:
# ══════════════════════════════════════════════
# ── TRAIN/TEST SPLIT — Feature extraction se PEHLE ──
# ══════════════════════════════════════════════

train_raw, test_raw = df_preprocessed.randomSplit([0.8, 0.2], seed=42)
train_raw.cache()
test_raw.cache()
print(f"Train rows: {train_raw.count()} | Test rows: {test_raw.count()}")

# ── TF-IDF: Sirf train par fit karo ──
cv_model    = cv.fit(train_raw)
train_tf    = cv_model.transform(train_raw)
test_tf     = cv_model.transform(test_raw)

idf_model   = idf.fit(train_tf)
train_tfidf = idf_model.transform(train_tf)
test_tfidf  = idf_model.transform(test_tf)

train_tfidf.cache()
test_tfidf.cache()
print("TF-IDF features ready.")

# ── Word2Vec: Sirf train par fit karo ──
w2v_model = word2vec.fit(train_raw)
train_w2v = w2v_model.transform(train_raw)
test_w2v  = w2v_model.transform(test_raw)

train_w2v.cache()
test_w2v.cache()
print("Word2Vec features ready.")

Train rows: 20081 | Test rows: 4997
TF-IDF features ready.
Word2Vec features ready.


In [20]:
import numpy as np
import urllib.request
import zipfile
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, VectorUDT

# Step 2.4.1: Download and extract the pre-trained GloVe embeddings
# We utilize the 100-dimensional Wikipedia 2014 + Gigaword 5 vectors.
print("Downloading GloVe embeddings...")
glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
urllib.request.urlretrieve(glove_url, "glove.6B.zip")

with zipfile.ZipFile("glove.6B.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# Step 2.4.2: Parse the GloVe text file into a local Python dictionary
glove_dict = {}
with open("glove.6B.100d.txt", 'r', encoding='utf-8') as f:
    for line in f:
      values = line.split()
      word = values[0]  # <--- Yahan 0 index add karein
      vector = np.asarray(values[1:], dtype='float32')
      glove_dict[word] = vector
# Step 2.4.3: Broadcast the dictionary to all Spark worker nodes
# Broadcasting prevents Spark from sending the massive dictionary repeatedly over the network for every task.
broadcast_glove = spark.sparkContext.broadcast(glove_dict)

# Step 2.4.4: Define the User Defined Function to average the word vectors
def average_glove_embeddings(words):
    vectors = [broadcast_glove.value[w] for w in words if w in broadcast_glove.value]
    if not vectors:
        # If no words in the review exist in the GloVe vocabulary, return a zero vector
        return Vectors.dense(np.zeros(100))
    # Compute the column-wise arithmetic mean across the sequence of word vectors [24]
    document_vector = np.mean(vectors, axis=0)
    return Vectors.dense(document_vector)

# Register the Python function as a Spark UDF that returns a Vector type
glove_udf = udf(average_glove_embeddings, VectorUDT())

# Step 2.4.5: Transform the DataFrame by applying the UDF
# GloVe — train aur test dono par apply karo
train_glove = train_raw.withColumn("glove_features", glove_udf("filtered_words"))
test_glove  = test_raw.withColumn("glove_features",  glove_udf("filtered_words"))

train_glove.cache()
test_glove.cache()
print("GloVe features ready.")

GloVe features ready.


In [21]:
# Step 2.5.1: Download the English FastText.vec file
# We use the 300-dimensional English word vectors trained on Wikipedia and Common Crawl.
print("Downloading FastText embeddings...")
fasttext_url = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip"
urllib.request.urlretrieve(fasttext_url, "wiki-news-300d-1M.vec.zip")

with zipfile.ZipFile("wiki-news-300d-1M.vec.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# Step 2.5.2: Optimized Parsing (Sirf zaroori words load karein)
# Pehle apne dataset ke unique words nikal lein
unique_words = set()
for row in df_preprocessed.select("filtered_words").collect():
    unique_words.update(row[0])

fasttext_dict = {}
with open("wiki-news-300d-1M.vec", 'r', encoding='utf-8') as f:
    next(f) # Header skip karein
    for line in f:
        values = line.rstrip().split(' ')
        word = values[0]
        # Sirf wahi words load karein jo dataset mein hain
        if word in unique_words:
            vector = np.asarray(values[1:], dtype='float32')
            fasttext_dict[word] = vector

# Ab broadcast memory mein fit ho jayega
broadcast_fasttext = spark.sparkContext.broadcast(fasttext_dict)

# Step 2.5.4: Define the UDF to average the 300-dimensional FastText vectors
def average_fasttext_embeddings(words):
    vectors = [broadcast_fasttext.value[w] for w in words if w in broadcast_fasttext.value]
    if not vectors:
        return Vectors.dense(np.zeros(300))
    document_vector = np.mean(vectors, axis=0)
    return Vectors.dense(document_vector)

fasttext_udf = udf(average_fasttext_embeddings, VectorUDT())

# Step 2.5.5: Transform the DataFrame
# FastText — train aur test dono par apply karo
train_fasttext = train_raw.withColumn("fasttext_features", fasttext_udf("filtered_words"))
test_fasttext  = test_raw.withColumn("fasttext_features",  fasttext_udf("filtered_words"))

train_fasttext.cache()
test_fasttext.cache()
print("FastText features ready.")

FastText features ready.


In [22]:
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf, col, udf
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.linalg import Vectors, VectorUDT
from transformers import BertTokenizer, BertModel
import torch

# ── BERT model driver par cache karo (workers mein re-load hoga) ──
BERT_MODEL_NAME = 'bert-base-uncased'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

_ = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
_ = BertModel.from_pretrained(BERT_MODEL_NAME)
print("BERT cached.")

# ── Pandas UDF define karo ──
@pandas_udf(ArrayType(FloatType()))
def get_bert_embeddings_udf(texts: pd.Series) -> pd.Series:
    from transformers import BertTokenizer, BertModel
    import torch, numpy as np

    _tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    _model     = BertModel.from_pretrained('bert-base-uncased')
    _model.eval()
    _device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    _model  = _model.to(_device)

    embeddings  = []
    batch_size  = 8
    texts_list  = texts.tolist()

    for i in range(0, len(texts_list), batch_size):
        batch   = texts_list[i : i + batch_size]
        encoded = _tokenizer(
            batch,
            return_tensors='pt',
            max_length=512,
            truncation=True,
            padding=True,
            add_special_tokens=True
        )
        encoded = {k: v.to(_device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = _model(**encoded)

        cls_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls_vecs.tolist())

    return pd.Series(embeddings)

# ── Array to MLlib Vector ──
array_to_vector_udf = udf(
    lambda arr: Vectors.dense(arr) if arr else Vectors.dense(np.zeros(768)),
    VectorUDT()
)

# ── BERT apply karo — review_bert column use karo ──
print("Generating BERT embeddings for train... (~15-20 min on GPU)")
train_bert = train_raw \
    .withColumn("bert_raw",   get_bert_embeddings_udf(col("review_bert"))) \
    .withColumn("bert_features", array_to_vector_udf("bert_raw")) \
    .drop("bert_raw")

print("Generating BERT embeddings for test...")
test_bert = test_raw \
    .withColumn("bert_raw",   get_bert_embeddings_udf(col("review_bert"))) \
    .withColumn("bert_features", array_to_vector_udf("bert_raw")) \
    .drop("bert_raw")

train_bert.cache()
test_bert.cache()
print("BERT features ready.")

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT cached.
Generating BERT embeddings for train... (~15-20 min on GPU)
Generating BERT embeddings for test...
BERT features ready.


In [23]:
from transformers import RobertaTokenizer, RobertaModel
from sentence_transformers import SentenceTransformer

# ══════════════════════════════════════════════
# ── RoBERTa embeddings ──
# ══════════════════════════════════════════════

ROBERTA_MODEL_NAME = 'roberta-base'
_ = RobertaTokenizer.from_pretrained(ROBERTA_MODEL_NAME)
_ = RobertaModel.from_pretrained(ROBERTA_MODEL_NAME)
print("RoBERTa cached.")

@pandas_udf(ArrayType(FloatType()))
def get_roberta_embeddings_udf(texts: pd.Series) -> pd.Series:
    from transformers import RobertaTokenizer, RobertaModel
    import torch, numpy as np

    _tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    _model     = RobertaModel.from_pretrained('roberta-base')
    _model.eval()
    _device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    _model  = _model.to(_device)

    embeddings = []
    batch_size = 8
    texts_list = texts.tolist()

    for i in range(0, len(texts_list), batch_size):
        batch = texts_list[i : i + batch_size]
        encoded = _tokenizer(
            batch,
            return_tensors='pt',
            max_length=512,
            truncation=True,
            padding=True
        )
        encoded = {k: v.to(_device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = _model(**encoded)

        # RoBERTa mein bhi pehla token (<s>) CLS jaisa role nibhata hai
        cls_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls_vecs.tolist())

    return pd.Series(embeddings)

print("Generating RoBERTa embeddings for train...")
train_roberta = train_raw \
    .withColumn("roberta_raw", get_roberta_embeddings_udf(col("review_bert"))) \
    .withColumn("roberta_features", array_to_vector_udf("roberta_raw")) \
    .drop("roberta_raw")

print("Generating RoBERTa embeddings for test...")
test_roberta = test_raw \
    .withColumn("roberta_raw", get_roberta_embeddings_udf(col("review_bert"))) \
    .withColumn("roberta_features", array_to_vector_udf("roberta_raw")) \
    .drop("roberta_raw")

train_roberta.cache()
test_roberta.cache()
print("RoBERTa features ready.")


# ══════════════════════════════════════════════
# ── SBERT embeddings ──
# ══════════════════════════════════════════════
# SBERT alag hai — ye direct sentence-level embedding deta hai (mean pooling
# already built-in hai), isliye CLS token nikalne ki zaroorat nahi.
# 'all-mpnet-base-v2' -> 768-dim (BERT/RoBERTa se compare karne ke liye best)

SBERT_MODEL_NAME = 'all-mpnet-base-v2'
_ = SentenceTransformer(SBERT_MODEL_NAME)
print("SBERT cached.")

@pandas_udf(ArrayType(FloatType()))
def get_sbert_embeddings_udf(texts: pd.Series) -> pd.Series:
    from sentence_transformers import SentenceTransformer
    import torch

    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    _model  = SentenceTransformer('all-mpnet-base-v2', device=_device)

    texts_list = texts.tolist()
    # SBERT apna internal batching + pooling khud handle karta hai
    embeddings = _model.encode(
        texts_list,
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    return pd.Series(embeddings.tolist())

print("Generating SBERT embeddings for train...")
train_sbert = train_raw \
    .withColumn("sbert_raw", get_sbert_embeddings_udf(col("review_bert"))) \
    .withColumn("sbert_features", array_to_vector_udf("sbert_raw")) \
    .drop("sbert_raw")

print("Generating SBERT embeddings for test...")
test_sbert = test_raw \
    .withColumn("sbert_raw", get_sbert_embeddings_udf(col("review_bert"))) \
    .withColumn("sbert_features", array_to_vector_udf("sbert_raw")) \
    .drop("sbert_raw")

train_sbert.cache()
test_sbert.cache()
print("SBERT features ready.")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT cached.
Generating SBERT embeddings for train...
Generating SBERT embeddings for test...
SBERT features ready.


In [24]:
from pyspark.ml.classification import (
    NaiveBayes, LogisticRegression,
    LinearSVC, MultilayerPerceptronClassifier
)

# TF-IDF classifiers
nb_classifier  = NaiveBayes(featuresCol="tfidf_features",    labelCol="label", smoothing=1.0, modelType="multinomial")
lr_classifier  = LogisticRegression(featuresCol="tfidf_features", labelCol="label", maxIter=20)
svm_classifier = LinearSVC(featuresCol="tfidf_features",     labelCol="label", maxIter=20)
ann_tfidf      = MultilayerPerceptronClassifier(featuresCol="tfidf_features",    labelCol="label", layers=[5000, 16, 2],    maxIter=10, seed=42)

# Embedding classifiers — har ek ka apna featuresCol
ann_w2v        = MultilayerPerceptronClassifier(featuresCol="word2vec_features", labelCol="label", layers=[100, 64, 32, 2], maxIter=50, seed=42)
ann_glove      = MultilayerPerceptronClassifier(featuresCol="glove_features",    labelCol="label", layers=[100, 64, 32, 2], maxIter=50, seed=42)
ann_fasttext   = MultilayerPerceptronClassifier(featuresCol="fasttext_features", labelCol="label", layers=[300, 64, 32, 2], maxIter=50, seed=42)
ann_bert       = MultilayerPerceptronClassifier(featuresCol="bert_features",     labelCol="label", layers=[768, 256, 64, 2],maxIter=50, stepSize=0.001, seed=42)
ann_roberta = MultilayerPerceptronClassifier(
    featuresCol="roberta_features", labelCol="label",
    layers=[768, 256, 64, 2], maxIter=50, stepSize=0.001, seed=42
)

ann_sbert = MultilayerPerceptronClassifier(
    featuresCol="sbert_features", labelCol="label",
    layers=[768, 128, 32, 2], maxIter=50, stepSize=0.001, seed=42
)

print("All classifiers defined.")

All classifiers defined.


In [25]:
def train_and_evaluate(model_name, classifier, train_df, test_df):
    """
    Executes distributed model training and computes classification metrics.
    """
    print(f"\n--- Initiating Training: {model_name} ---")

    # Start the clock
    start_time = time.time()

    # Execute model fitting
    trained_model = classifier.fit(train_df)

    # Stop the clock
    end_time = time.time()
    training_time = round(end_time - start_time, 2)
    print(f"Training Duration: {training_time} seconds")

    # Generate inference predictions
    predictions = trained_model.transform(test_df)

    # Compute fractional metrics
    accuracy = eval_accuracy.evaluate(predictions)
    precision = eval_precision.evaluate(predictions)
    recall = eval_recall.evaluate(predictions)
    f1 = eval_f1.evaluate(predictions)

    # --- FIXED: Confusion Matrix Extraction ---
    # Prediction aur Label ko float mein convert karke RDD banana (Function ke andar hona chahiye)
    prediction_and_label = predictions.select("prediction", "label").rdd.map(lambda row: (float(row[0]), float(row[1])))

    # MulticlassMetrics legacy object
    metrics = MulticlassMetrics(prediction_and_label)

    # Extract confusion matrix
    cm = metrics.confusionMatrix().toArray()

    print(f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}")
    print("Confusion Matrix:\n", cm)

    # Return results dictionary
    return {
        "Combination": model_name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "Training_Time (sec)": training_time
    }

In [26]:
import time
import pandas as pd
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Evaluators initialize karein (agar pehle nahi kiye hain)
eval_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
eval_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
eval_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
eval_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

In [27]:
results_list = []

# TF-IDF combinations — train_tfidf / test_tfidf use karo
results_list.append(train_and_evaluate("TF-IDF + Naive Bayes",        nb_classifier,  train_tfidf,    test_tfidf))
results_list.append(train_and_evaluate("TF-IDF + Logistic Regression",lr_classifier,  train_tfidf,    test_tfidf))
results_list.append(train_and_evaluate("TF-IDF + SVM",                svm_classifier, train_tfidf,    test_tfidf))
results_list.append(train_and_evaluate("TF-IDF + ANN",                ann_tfidf,      train_tfidf,    test_tfidf))

# Embedding combinations — har ek ka apna dataset
results_list.append(train_and_evaluate("Word2Vec + ANN",              ann_w2v,        train_w2v,      test_w2v))
results_list.append(train_and_evaluate("GloVe + ANN",                 ann_glove,      train_glove,    test_glove))
results_list.append(train_and_evaluate("FastText + ANN",              ann_fasttext,   train_fasttext, test_fasttext))
results_list.append(train_and_evaluate("BERT + ANN", ann_bert, train_bert, test_bert))
train_bert.unpersist(); test_bert.unpersist()

results_list.append(train_and_evaluate("RoBERTa + ANN", ann_roberta, train_roberta, test_roberta))
train_roberta.unpersist(); test_roberta.unpersist()

results_list.append(train_and_evaluate("SBERT + ANN", ann_sbert, train_sbert, test_sbert))
train_sbert.unpersist(); test_sbert.unpersist()
# Final results table
df_results = spark.createDataFrame(pd.DataFrame(results_list))
print("\n--- Final Results ---")
df_results.show(truncate=False)


--- Initiating Training: TF-IDF + Naive Bayes ---
Training Duration: 6.65 seconds


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Accuracy: 0.8515 | Precision: 0.8517 | Recall: 0.8515 | F1-Score: 0.8515
Confusion Matrix:
 [[2132.  347.]
 [ 395. 2123.]]

--- Initiating Training: TF-IDF + Logistic Regression ---
Training Duration: 3.8 seconds
Accuracy: 0.8427 | Precision: 0.8427 | Recall: 0.8427 | F1-Score: 0.8427
Confusion Matrix:
 [[2092.  387.]
 [ 399. 2119.]]

--- Initiating Training: TF-IDF + SVM ---
Training Duration: 4.04 seconds
Accuracy: 0.8717 | Precision: 0.8717 | Recall: 0.8717 | F1-Score: 0.8717
Confusion Matrix:
 [[2152.  327.]
 [ 314. 2204.]]

--- Initiating Training: TF-IDF + ANN ---
Training Duration: 16.66 seconds
Accuracy: 0.8841 | Precision: 0.8842 | Recall: 0.8841 | F1-Score: 0.8841
Confusion Matrix:
 [[2206.  273.]
 [ 306. 2212.]]

--- Initiating Training: Word2Vec + ANN ---
Training Duration: 15.21 seconds
Accuracy: 0.8507 | Precision: 0.8514 | Recall: 0.8507 | F1-Score: 0.8506
Confusion Matrix:
 [[2050.  429.]
 [ 317. 2201.]]

--- Initiating Training: GloVe + ANN ---
Training Duration: 21.58

In [28]:
# Convert the collected Python dictionaries into a finalized DataFrame for tabular display
df_results = spark.createDataFrame(pd.DataFrame(results_list))

print("\n--- Final Aggregated Conclusion Table ---")
# Truncate=False ensures the column boundaries fully render the metric data
df_results.show(truncate=False)


--- Final Aggregated Conclusion Table ---
+----------------------------+--------+---------+------+--------+-------------------+
|Combination                 |Accuracy|Precision|Recall|F1-Score|Training_Time (sec)|
+----------------------------+--------+---------+------+--------+-------------------+
|TF-IDF + Naive Bayes        |0.8515  |0.8517   |0.8515|0.8515  |6.65               |
|TF-IDF + Logistic Regression|0.8427  |0.8427   |0.8427|0.8427  |3.8                |
|TF-IDF + SVM                |0.8717  |0.8717   |0.8717|0.8717  |4.04               |
|TF-IDF + ANN                |0.8841  |0.8842   |0.8841|0.8841  |16.66              |
|Word2Vec + ANN              |0.8507  |0.8514   |0.8507|0.8506  |15.21              |
|GloVe + ANN                 |0.8023  |0.8025   |0.8023|0.8023  |21.58              |
|FastText + ANN              |0.8431  |0.8431   |0.8431|0.8431  |28.52              |
|BERT + ANN                  |0.8755  |0.8755   |0.8755|0.8755  |680.05             |
|RoBERTa + 